In [ ]:
import re
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122 Safari/537.36"
BASE = "https://bazarstore.az"

ALLOWED_AVAIL = {"in_stock", "out_of_stock", "unknown"}


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def make_run_id(market: str) -> str:
    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    return f"{market}_{ts}"

def _set_page(url: str, page: int) -> str:
    parts = urlparse(url)
    qs = parse_qs(parts.query)
    qs["page"] = [str(page)]
    new_query = urlencode(qs, doseq=True)
    return urlunparse((parts.scheme, parts.netloc, parts.path, parts.params, new_query, parts.fragment))

def _guess_category_slug(url: str) -> str:
    parts = urlparse(url)
    path = parts.path.strip("/").split("/")
    if len(path) >= 2 and path[0] == "collections":
        return path[1]
    if "search" in path:
        return "search"
    return "unknown"

def _guess_search_query(url: str):
    qs = parse_qs(urlparse(url).query)
    return qs.get("q", [None])[0]

def _guess_location_id(url: str):
    qs = parse_qs(urlparse(url).query)
    key = "filter.p.m.custom.lokasyonlar"
    return qs.get(key, [None])[0]

def _abs_url(href: str) -> str:
    if not href:
        return ""
    if href.startswith("http"):
        return href
    if href.startswith("/"):
        return BASE + href
    return BASE + "/" + href

def _parse_price_azn(text: str):
    if not text:
        return None
    t = text.replace("\xa0", " ").strip()
    m = re.search(r"(\d+(?:[.,]\d+)?)", t)
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", "."))
    except:
        return None

def _product_handle_from_url(product_url: str) -> str | None:
    # https://bazarstore.az/products/<handle>
    try:
        parts = urlparse(product_url)
        path = parts.path.strip("/").split("/")
        if len(path) >= 2 and path[0] == "products":
            return path[1]
    except:
        pass
    return None

def _safe_get_json(session: requests.Session, url: str, timeout=30, retries=3, sleep_base=0.6):
    last_err = None
    for i in range(retries):
        try:
            r = session.get(url, headers={"User-Agent": UA}, timeout=timeout)
            if r.status_code in (429, 503):
                time.sleep(sleep_base * (2 ** i))
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(sleep_base * (2 ** i))
    raise last_err


def _extract_products_from_html(html: str):
    soup = BeautifulSoup(html, "html.parser")
    anchors = soup.select('a[href*="/products/"]')
    if not anchors:
        return []

    rows = []
    for a in anchors:
        href = a.get("href")
        if not href or "/products/" not in href:
            continue

        product_url = _abs_url(href)

        title = a.get_text(" ", strip=True) or None
        if not title:
            title = a.get("aria-label") or a.get("title") or None

        card = a.find_parent(["article", "li", "div"])
        card_text = card.get_text(" ", strip=True) if card else ""

        
        price_texts = []
        if card:
            for t in card.find_all(string=re.compile(r"(₼|AZN)", re.I)):
                price_texts.append(str(t))
        if not price_texts and (("₼" in card_text) or ("AZN" in card_text.upper())):
            price_texts = [card_text]

        prices = []
        for pt in price_texts:
            p = _parse_price_azn(pt)
            if p is not None:
                prices.append(p)

        price_current = None
        price_old = None
        if prices:
            price_current = float(min(prices))
            mx = float(max(prices))
            price_old = mx if mx != price_current else None

        low = (card_text or "").lower()
        if any(x in low for x in ["sold out", "tükəndi", "stokda yoxdur", "out of stock", "mövcud deyil"]):
            availability = "out_of_stock"
        else:
            availability = "in_stock"

        rows.append({
            "product_url": product_url,
            "product_handle": _product_handle_from_url(product_url),
            "product_name_raw": title,
            "price_current_azn": price_current,
            "price_old_azn": price_old,
            "availability_status": availability,
        })

    
    dedup = {}
    for r in rows:
        dedup[r["product_url"]] = r
    return list(dedup.values())


def scrape_urls_to_df3(urls, max_pages=40, sleep_sec=0.5, timeout_sec=30, stop_after_empty_pages=1):
    sess = requests.Session()
    all_rows = []

    for url in urls:
        category_raw = _guess_category_slug(url)
        location_id = _guess_location_id(url)
        search_query = _guess_search_query(url)

        empty_pages = 0
        for page in range(1, max_pages + 1):
            page_url = _set_page(url, page)

            r = sess.get(page_url, headers={"User-Agent": UA}, timeout=timeout_sec)
            r.raise_for_status()

            products = _extract_products_from_html(r.text)

            if not products:
                empty_pages += 1
                if empty_pages >= stop_after_empty_pages:
                    break
            else:
                empty_pages = 0

            for p in products:
                p["category_raw"] = category_raw
                p["location_id"] = location_id
                p["search_query"] = search_query
                p["source_url"] = url
                p["page_no"] = page
                p["page_url"] = page_url
                all_rows.append(p)

            time.sleep(sleep_sec)

    return pd.DataFrame(all_rows)



def add_sku_barcode_from_product_js(df: pd.DataFrame, sleep_sec=0.25, timeout_sec=30):
    """
    Shopify endpoint: https://bazarstore.az/products/<handle>.js
    Ordan variants[0].sku və variants[0].barcode götürürük.
    Qeyd: Bəzi mağazalarda barcode/sku boş ola bilər.
    """
    if df.empty:
        return df

    sess = requests.Session()
    cache = {}  
    sku_list = []
    barcode_list = []
    variant_id_list = []
    variant_title_list = []
    product_id_list = []

    for _, row in df.iterrows():
        handle = row.get("product_handle")
        if not handle or not isinstance(handle, str):
            sku_list.append(None); barcode_list.append(None)
            variant_id_list.append(None); variant_title_list.append(None); product_id_list.append(None)
            continue

        if handle in cache:
            sku, bc, vid, vtitle, pid = cache[handle]
            sku_list.append(sku); barcode_list.append(bc)
            variant_id_list.append(vid); variant_title_list.append(vtitle); product_id_list.append(pid)
            continue

        js_url = f"{BASE}/products/{handle}.js"
        try:
            j = _safe_get_json(sess, js_url, timeout=timeout_sec, retries=3, sleep_base=0.6)
            pid = j.get("id")
            variants = j.get("variants") or []
            v0 = variants[0] if variants else {}
            sku = v0.get("sku")
            bc = v0.get("barcode") 
            vid = v0.get("id")
            vtitle = v0.get("public_title") or v0.get("title")

            cache[handle] = (sku, bc, vid, vtitle, pid)
            sku_list.append(sku); barcode_list.append(bc)
            variant_id_list.append(vid); variant_title_list.append(vtitle); product_id_list.append(pid)

        except Exception:
            cache[handle] = (None, None, None, None, None)
            sku_list.append(None); barcode_list.append(None)
            variant_id_list.append(None); variant_title_list.append(None); product_id_list.append(None)

        time.sleep(sleep_sec)

    out = df.copy()
    out["shopify_product_id"] = product_id_list
    out["shopify_variant_id"] = variant_id_list
    out["variant_title"] = variant_title_list
    out["sku"] = sku_list
    out["barcode"] = barcode_list  
    return out


UNIT_PATTERN = re.compile(
    r'(?P<qty>\d+(?:[.,]\d+)?)\s*(?P<unit>kg|kq|g|gr|qram|ml|l|lt|litr|ədəd|edet|sht)',
    flags=re.IGNORECASE
)

def extract_unit_from_name(name: str):
    if not isinstance(name, str):
        return None, None
    m = UNIT_PATTERN.search(name.lower())
    if not m:
        return None, None

    qty = float(m.group("qty").replace(",", "."))
    unit = m.group("unit").lower()

    if unit in ["kg", "kq"]:
        return qty * 1000, "g"
    if unit in ["g", "gr", "qram"]:
        return qty, "g"
    if unit in ["l", "lt", "litr"]:
        return qty * 1000, "ml"
    if unit == "ml":
        return qty, "ml"
    if unit in ["ədəd", "edet", "sht"]:
        return qty, "unit"

    return None, None


def enrich_and_harden_df3(df3: pd.DataFrame, market: str, snapshot_date: str, run_id: str) -> pd.DataFrame:
    df3 = df3.copy()

    df3["scrape_run_id"] = run_id
    df3["scrape_ts_utc"] = pd.Timestamp.utcnow()
    df3["snapshot_date"] = pd.to_datetime(snapshot_date, errors="coerce").date().isoformat()
    df3["market"] = market

    df3["price_current_azn"] = pd.to_numeric(df3["price_current_azn"], errors="coerce")
    df3["price_old_azn"] = pd.to_numeric(df3["price_old_azn"], errors="coerce")

    df3["promo_flag"] = ((~df3["price_old_azn"].isna()) & (df3["price_old_azn"] > df3["price_current_azn"])).astype(int)

    
    s = df3["availability_status"].astype("string").str.strip().str.lower()
    df3["availability_status"] = np.where(
        s.isin(["in_stock", "available", "true", "1", "yes"]),
        "in_stock",
        np.where(
            s.isin(["out_of_stock", "unavailable", "false", "0", "no"]),
            "out_of_stock",
            "unknown"
        )
    )

    
    df3["purchasable_balance"] = np.where(
        df3["availability_status"].eq("in_stock"), 1,
        np.where(df3["availability_status"].eq("out_of_stock"), 0, np.nan)
    )
    df3["purchasable_balance_proxy"] = df3["purchasable_balance"]
    df3["pb_source"] = "html_availability_proxy"

   
    df3[["net_qty_base", "unit_raw"]] = df3["product_name_raw"].apply(lambda x: pd.Series(extract_unit_from_name(x)))
    df3["net_qty_base"] = pd.to_numeric(df3["net_qty_base"], errors="coerce")

    df3["unit_price"] = df3["price_current_azn"] / df3["net_qty_base"]

    
    df3["dq_invalid_price"] = df3["price_current_azn"].isna() | (df3["price_current_azn"] <= 0)
    df3["dq_missing_unit"] = df3["net_qty_base"].isna() | (df3["net_qty_base"] <= 0)
    df3["dq_invalid_availability"] = ~df3["availability_status"].isin(list(ALLOWED_AVAIL))

    def build_product_key(row):
        bc = str(row.get("barcode") or "").strip()
        sku = str(row.get("sku") or "").strip()
        if bc:
            return f"barcode|{bc}"
        if sku:
            return f"sku|{sku}"
        parts = [
            str(row.get("product_name_raw", "")).lower().strip(),
            str(row.get("net_qty_base", "")),
            str(row.get("unit_raw", "")),
            str(row.get("location_id", "")),
        ]
        return "name|"+ "|".join(parts)

    df3["product_key"] = df3.apply(build_product_key, axis=1)
    df3["product_key_hash"] = pd.util.hash_pandas_object(df3["product_key"], index=False).astype("string")

   
    df3 = df3.sort_values("scrape_ts_utc")
    df3 = df3.drop_duplicates(
        subset=["market", "location_id", "product_key_hash", "snapshot_date"],
        keep="last"
    )

    return df3


def write_bronze_csv(df3: pd.DataFrame, market: str, snapshot_date: str, run_id: str, base_dir="data") -> Path:
    out_dir = Path(base_dir) / "bronze" / f"market={market}" / f"snapshot_date={snapshot_date}" / f"run_id={run_id}"
    ensure_dir(out_dir)
    out_path = out_dir / "snapshot.csv"
    df3.to_csv(out_path, index=False, encoding="utf-8-sig")
    return out_path


def run_bazarstore_bronze(urls, snapshot_date=None, base_dir="data", enrich_sku_barcode=True):
    market = "bazarstore"
    run_id = make_run_id(market)

    if snapshot_date is None:
        snapshot_date = datetime.now(timezone.utc).date().isoformat()

    
    df = scrape_urls_to_df3(urls, max_pages=40, sleep_sec=0.5, timeout_sec=30, stop_after_empty_pages=1)
    if df.empty:
        raise RuntimeError("Scrape 0 rows qaytardı. Selector / blok / filter check lazımdır.")

    
    if enrich_sku_barcode:
        df = add_sku_barcode_from_product_js(df, sleep_sec=0.25, timeout_sec=30)

  
    df3 = enrich_and_harden_df3(df, market=market, snapshot_date=snapshot_date, run_id=run_id)

    
    out = write_bronze_csv(df3, market=market, snapshot_date=snapshot_date, run_id=run_id, base_dir=base_dir)

    print("✅ DONE")
    print("run_id:", run_id)
    print("rows:", len(df3))
    print("sku_count:", df3["product_key_hash"].nunique())
    print("saved:", out)
    print("barcode filled:", int(df3["barcode"].notna().sum()) if "barcode" in df3.columns else 0)
    print("sku filled:", int(df3["sku"].notna().sum()) if "sku" in df3.columns else 0)

    return df3, {"run_id": run_id, "snapshot_path": str(out), "snapshot_date": snapshot_date}


urls = [
    "https://bazarstore.az/collections/sud?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/qatiq?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/k%C9%99r%C9%99-yagi?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yumurta?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yag?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/duyu?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/makaron?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/duz?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/un?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/s%C9%99k%C9%99r-tozu?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/q%C9%99nd?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yuyucu-toz-avtomat?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yuyucu-toz-%C9%99l?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yuyucu-gel?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102"
]

df3, meta = run_bazarstore_bronze(urls, snapshot_date="2025-02-13", base_dir="data", enrich_sku_barcode=True)
df3.head(10)


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_18420\948465712.py:309: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  df3["scrape_ts_utc"] = pd.Timestamp.utcnow()


✅ DONE
run_id: bazarstore_20260213_195641
rows: 503
sku_count: 503
saved: data\bronze\market=bazarstore\snapshot_date=2025-02-13\run_id=bazarstore_20260213_195641\snapshot.csv
barcode filled: 503
sku filled: 503


,product_url,product_handle,product_name_raw,price_current_azn,price_old_azn,availability_status,category_raw,location_id,search_query,source_url,...,purchasable_balance_proxy,pb_source,net_qty_base,unit_raw,unit_price,dq_invalid_price,dq_missing_unit,dq_invalid_availability,product_key,product_key_hash
0,https://bazarstore.az/products/milla-sud-1-l-3...,milla-sud-1-l-3-2-yagli,"MİLLA SÜD 1 L 3,2% YAĞLI Normal qiymət 2.49 ₼ ...",2.49,2.8,in_stock,sud,4102,None,https://bazarstore.az/collections/sud?filter.v...,...,1.0,html_availability_proxy,1000.0,ml,0.00249,False,False,False,barcode|4760081600037,18435638276036255331
352,https://bazarstore.az/products/ankara-vit-maka...,ankara-vit-makaron-500-q-tel-vermisel,ANKARA VİT.MAKARON 500 Q TEL VERMİŞEL Normal q...,1.90,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|8690576529818,9165545087904699026
351,https://bazarstore.az/products/barilla-makaron...,barilla-makaron-450-q-bavette-n13,BARILLA MAKARON 450 Q BAVETTE N13 Normal qiymə...,2.45,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|8076809575942,7385070183523185716
350,https://bazarstore.az/products/barilla-makaron...,barilla-makaron-450-q-cellentani,BARILLA MAKARON 450 Q CELLENTANI Normal qiymət...,2.45,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|8076809575966,12572065519880091627
349,https://bazarstore.az/products/filiz-makaron-5...,filiz-makaron-500-q-burgu,FİLİZ MAKARON 500 Q BURGU Normal qiymət 1.75 ₼...,1.75,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|8690579081931,2581497115771324701
348,https://bazarstore.az/products/harika-makaron-...,harika-makaron-500-q-vermisel,HARIKA MAKARON 500 Q VERMİŞEL Normal qiymət 1....,1.69,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|4760013412097,16690101042824279418
347,https://bazarstore.az/products/ankara-makaron-...,ankara-makaron-500-q-linguine,ANKARA MAKARON 500 Q LINGUINE Normal qiymət 1....,1.90,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|8690576029035,4273619524002126766
346,https://bazarstore.az/products/barilla-makaron...,barilla-makaron-450-q-spaghettoni-n7,BARILLA MAKARON 450 Q SPAGHETTONI N7 Normal qi...,2.45,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|8076809576116,9810709753969053149
345,https://bazarstore.az/products/barilla-makaron...,barilla-makaron-450-q-spaghettini-n3,BARILLA MAKARON 450 Q SPAGHETTINI N3 Normal qi...,2.45,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|8076809576109,11992355026147623071
344,https://bazarstore.az/products/ankara-vit-maka...,ankara-vit-makaron-500-q-burgu,ANKARA VİT.MAKARON 500 Q BURGU Normal qiymət 1...,1.90,NaN,in_stock,makaron,4102,None,https://bazarstore.az/collections/makaron?filt...,...,1.0,html_availability_proxy,NaN,NaN,NaN,False,True,False,barcode|8690576529177,17123739243091783320


In [ ]:

def write_bronze_csv(df3: pd.DataFrame, market: str, snapshot_date: str, run_id: str, base_dir="data") -> Path:
    out_dir = Path(base_dir) / "bronze" / f"market={market}" / f"snapshot_date={snapshot_date}" / f"run_id={run_id}"
    ensure_dir(out_dir)
    out_path = out_dir / "snapshot.csv"
    df3.to_csv(out_path, index=False, encoding="utf-8-sig")
    return out_path


def run_bazarstore_bronze(urls, snapshot_date=None, base_dir="data", enrich_sku_barcode=True):
    market = "bazarstore"
    run_id = make_run_id(market)

    if snapshot_date is None:
        snapshot_date = datetime.now(timezone.utc).date().isoformat()

    
    df = scrape_urls_to_df3(urls, max_pages=40, sleep_sec=0.5, timeout_sec=30, stop_after_empty_pages=1)
    if df.empty:
        raise RuntimeError("Scrape 0 rows qaytardı. Selector / blok / filter check lazımdır.")

   
    if enrich_sku_barcode:
        df = add_sku_barcode_from_product_js(df, sleep_sec=0.25, timeout_sec=30)

    
    df3 = enrich_and_harden_df3(df, market=market, snapshot_date=snapshot_date, run_id=run_id)

    
    out = write_bronze_csv(df3, market=market, snapshot_date=snapshot_date, run_id=run_id, base_dir=base_dir)

    print("✅ DONE")
    print("run_id:", run_id)
    print("rows:", len(df3))
    print("sku_count:", df3["product_key_hash"].nunique())
    print("saved:", out)
    print("barcode filled:", int(df3["barcode"].notna().sum()) if "barcode" in df3.columns else 0)
    print("sku filled:", int(df3["sku"].notna().sum()) if "sku" in df3.columns else 0)

    return df3, {"run_id": run_id, "snapshot_path": str(out), "snapshot_date": snapshot_date}


urls = [
    "https://bazarstore.az/collections/sud?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/qatiq?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/k%C9%99r%C9%99-yagi?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yumurta?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yag?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/duyu?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/makaron?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/duz?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/un?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/s%C9%99k%C9%99r-tozu?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/q%C9%99nd?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yuyucu-toz-avtomat?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yuyucu-toz-%C9%99l?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102",
    "https://bazarstore.az/collections/yuyucu-gel?filter.v.price.gte=&filter.v.price.lte=&filter.v.availability=1&filter.p.m.custom.lokasyonlar=4102"
]

df3, meta = run_bazarstore_bronze(urls, snapshot_date="2025-02-13", base_dir="data", enrich_sku_barcode=True)
df3.head(10)